In [ ]:
# Constraint Following & Format Robustness

## Goal

Evaluate how reliably a language model follows strict output constraints.

We test:
- Greedy vs Sampling
- Low vs High temperature
- Template prompts vs rule-only prompts

We measure:
- Format violations
- Number of bullets
- Extra text before/after
- Structure drift

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

In [ ]:
prompt_rules = """
Explain photosynthesis.

Rules:
- Output exactly 5 bullet points.
- Each bullet must start with "- ".
- Each bullet must contain at most 12 words.
- No extra text before or after.
""".strip()

In [ ]:
prompt_template = """
Explain photosynthesis.

Return ONLY this template:

- Bullet 1:
- Bullet 2:
- Bullet 3:
- Bullet 4:
- Bullet 5:

Replace each line with one short sentence (<=12 words).
Do not add extra text.
""".strip()

In [ ]:
def generate(prompt, do_sample=True, temperature=1.0, seed=0):
    set_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=do_sample,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
print("=== RULE PROMPT - GREEDY ===\n")
print(generate(prompt_rules, do_sample=False))

In [ ]:
print("\n=== RULE PROMPT - SAMPLING ===\n")

for i in range(3):
    print(f"\nRun {i}")
    print(generate(prompt_rules, do_sample=True, temperature=1.0, seed=i))

In [ ]:
print("=== TEMPLATE PROMPT - GREEDY ===\n")
print(generate(prompt_template, do_sample=False))

In [ ]:
print("\n=== TEMPLATE PROMPT - SAMPLING ===\n")

for i in range(3):
    print(f"\nRun {i}")
    print(generate(prompt_template, do_sample=True, temperature=1.0, seed=i))

In [11]:
def count_bullets(text):
    lines = text.splitlines()
    bullets = [l for l in lines if l.strip().startswith("- ")]
    return len(bullets)

print("\n=== Violation Check Example ===")
out = generate(prompt_template, do_sample=True, temperature=1.0, seed=0)
print("Bullet count:", count_bullets(out))


=== Violation Check Example ===
Bullet count: 6


In [ ]:
## Observations

Rule-based prompts:
- Often include extra text.
- May produce more than 5 bullets.
- Sampling increases violations.

Template prompts:
- Stronger structural adherence.
- Still not perfectly guaranteed.
- Sampling can still cause drift.

## Key Insight

Language models optimize probability, not compliance.

Constraints written in prompts are:
- Influential
- But not binding

Format robustness depends on:
- Decoding strategy
- Prompt structure
- Model alignment strength

In production systems, strict structure often requires:
- Post-processing
- Validation
- Re-asking or correction loops
